In [ ]:
# ============================================================
# CELL 1.2: CHUNKING & CORPUS PROCESSING
# ============================================================

def split_into_sentences(text: str) -> list:
    """Split text into sentences"""
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if s.strip()]

def chunk_by_sentences(text: str, file_path: str) -> list:
    """
    Chunk text by sentences with cleaning pipeline.
    Returns list of tuples: (chunk_uid, raw_text, text_for_scoring, sentence_count)
    """
    sentences = split_into_sentences(text)
    
    sentences_per_chunk = CONFIG["chunking"]["sentences_per_chunk"]
    min_sentences = CONFIG["chunking"]["min_sentences_to_keep"]
    
    chunks = []
    chunk_idx = 0
    
    for i in range(0, len(sentences), sentences_per_chunk):
        sent_slice = sentences[i:i + sentences_per_chunk]
        
        if len(sent_slice) < min_sentences:
            continue
        
        raw_text = " ".join(sent_slice)
        
        # Apply English filtering
        if CONFIG["chunking"]["drop_likely_english"]:
            filtered_sents = [s for s in sent_slice if not likely_english_sentence(s)]
            if len(filtered_sents) < min_sentences:
                text_for_scoring = ""
            else:
                text_for_scoring = normalize_space(" ".join(filtered_sents))
        else:
            text_for_scoring = normalize_space(raw_text)
        
        # Apply stopword removal
        if CONFIG["chunking"]["remove_stopwords"] and text_for_scoring:
            text_for_scoring = remove_stopwords_and_numbers(text_for_scoring)
        
        # Apply stemming
        if CONFIG["chunking"]["use_stemming"] and text_for_scoring:
            text_for_scoring = stem_text(text_for_scoring)
        
        text_for_scoring = normalize_space(text_for_scoring)
        
        chunks.append((
            make_chunk_uid(file_path, chunk_idx),
            raw_text,
            text_for_scoring,
            len(sent_slice)
        ))
        chunk_idx += 1
    
    return chunks

# Process corpus
print(f"\n{'='*60}")
print("PROCESSING CORPUS")
print(f"{'='*60}")

all_chunks = []
corpus_dir = Path(CONFIG["paths"]["corpus_dir"])

if not corpus_dir.exists():
    print(f"⚠ Corpus directory not found: {corpus_dir}")
else:
    doc_files = list(corpus_dir.glob("*.txt"))
    print(f"\nFound {len(doc_files)} documents")
    
    for doc_path in tqdm(doc_files, desc="Chunking documents"):
        with open(doc_path, 'r', encoding='utf-8', errors='ignore') as f:
            text = f.read()
        
        doc_chunks = chunk_by_sentences(text, str(doc_path))
        for chunk_uid, raw_text, text_for_scoring, sentence_count in doc_chunks:
            all_chunks.append({
                'file_path': str(doc_path),
                'chunk_uid': chunk_uid,
                'raw_text': raw_text,
                'text_for_scoring': text_for_scoring,
                'sentence_count': sentence_count
            })
    
    # Create DataFrame
    chunks_df = pd.DataFrame(all_chunks)
    
    # Save to Other_data
    fs.save_data(chunks_df, "chunked_corpus", "Other_data", "csv")
    
    print(f"\n✓ Processed {len(chunks_df)} chunks from {chunks_df['file_path'].nunique()} documents")
    print(f"  Avg sentences per chunk: {chunks_df['sentence_count'].mean():.1f}")
    print(f"  Chunks with empty scoring text: {(chunks_df['text_for_scoring'] == '').sum()}")
    
    # Save checkpoint
    fs.save_config("checkpoint1_chunks")

In [ ]:
# ============================================================
# CELL 1.1: TEXT CLEANING UTILITIES
# ============================================================

stemmer = SnowballStemmer("dutch")

# Build comprehensive stopword set
nltk_stopwords = set(stopwords.words('dutch')) | set(stopwords.words('english'))
custom_stopwords = set([
    "de","het","een","en","van","in","op","met","voor","tegen","zonder","bij",
    "naar","tot","uit","door","aan","om","te","als","ook","maar","want","dus",
    "of","dan","nog","wel","zijn","is","was","waren","worden","hebben","heeft",
    "had","doet","doen","al","alle","meer","minder","veel","weinig","binnen",
    "buiten","tussen","onder","boven","over","na","achter","naast","sinds",
    "tijdens","zoals","ik","jij","hij","zij","wij","jullie","u","je","ze",
    "dit","dat","die","deze","welke","ons","hun","hem","haar","bijlage",
    "bijlagen","inleiding","samenvatting","conclusie","conclusies","jaar","jaren",
])
ALL_STOPWORDS = nltk_stopwords | custom_stopwords

# Detection sets
ENGLISH_HINTS = set("the and of to in that is for on with as by from at it this be are were was has have will would can could should".split())
DUTCH_HINTS = set("de het een en van voor met op aan te is zijn worden was waren niet bij in over uit door naar tot als ook om".split())

def likely_english_sentence(s: str) -> bool:
    """Heuristic to detect likely English sentences"""
    if not CONFIG["chunking"]["drop_likely_english"]:
        return False
    tokens = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ]+", s.lower())
    if not tokens:
        return False
    e = sum(t in ENGLISH_HINTS for t in tokens)
    d = sum(t in DUTCH_HINTS for t in tokens)
    return e > max(2, d + 1)

def remove_stopwords_and_numbers(text: str) -> str:
    """Remove stopwords and digits"""
    if pd.isna(text):
        return ""
    tokens = re.findall(r"\b\w+\b", text.lower())
    filtered = [tok for tok in tokens if tok not in ALL_STOPWORDS and not tok.isdigit()]
    return " ".join(filtered)

def stem_text(text: str) -> str:
    """Apply Dutch stemming"""
    tokens = re.findall(r"\b\w+\b", text.lower())
    return " ".join(stemmer.stem(w) for w in tokens)

def normalize_space(text: str) -> str:
    """Normalize whitespace"""
    return re.sub(r"\s+", " ", text).strip()

def short_file_hash(path: str, n=8) -> str:
    """Create short hash for file identification"""
    return hashlib.sha1(path.encode("utf-8", errors="ignore")).hexdigest()[:n]

def make_chunk_uid(file_path: str, chunk_idx: int) -> str:
    """Generate unique chunk identifier"""
    return f"{short_file_hash(file_path)}:{chunk_idx:05d}"

print("✓ Text cleaning utilities ready")

---
# CHECKPOINT 1: Text Processing
---

This checkpoint:
- Loads corpus documents
- Chunks text into sentence-based segments  
- Applies cleaning (stopwords, English filtering, stemming)
- Saves chunks to Other_data/

**Resume from here**: If you already have `chunked_corpus.csv`, skip to CHECKPOINT 2

# Dictionary Discovery Workflow v3 - Structured

## Overview
This notebook implements a systematic dictionary-based topic discovery and model training workflow.

### Workflow Structure:
- **CHECKPOINT 0**: Initial Setup (Config & Folders)
- **CHECKPOINT 1**: Text Processing (Chunking)
- **CHECKPOINT 2**: Vocabulary Building
- **CHECKPOINT 3**: Dictionary Expansion → **MANUAL CURATION REQUIRED**
- **CHECKPOINT 4**: Topic Vector Creation
- **CHECKPOINT 5**: Chunk Scoring (Cosine Similarity)
- **CHECKPOINT 6**: Training Data Preparation
- **CHECKPOINT 7**: Model Training (BERTJE Fine-tuning)
- **CHECKPOINT 8**: Visualizations

### Folder Structure:
```
workflow_data/
  {ModelType}-{Topic}-{Date}-{Version}/
    ├── config/
    ├── Dictionary/
    │   └── Dictionary_suggestions/
    ├── Model_finetuning/
    ├── Cosine_labeling/
    ├── Bertje_labeling/
    ├── Visuals/
    └── Other_data/
```

---
# CHECKPOINT 0: Initial Setup
---

In [ ]:
# ============================================================
# CELL 0.1: IMPORTS
# ============================================================
import os
import re
import json
import hashlib
import shutil
import warnings
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm import tqdm

# NLTK
import nltk
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords")
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

# ML libraries
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer

# Suppress warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

print("✓ All imports successful")

In [ ]:
# ============================================================
# CELL 0.2: CONFIGURATION
# ============================================================

CONFIG = {
    # =====================
    # WORKFLOW METADATA
    # =====================
    "workflow": {
        # Model type: "Pretrained" or "Finetuned_{source_topic}"
        # Examples: "Pretrained", "Finetuned_Slavery", "Finetuned_Policy"
        "model_type": "Pretrained",
        
        # Topic being analyzed (can be combination like "Slavery-Policy")
        "topic": "Slavery",
        
        # Version (will auto-increment if not specified)
        "version": None,  # Set to None for auto-increment, or "v1", "v2", etc.
    },
    
    # =====================
    # PATHS
    # =====================
    "paths": {
        "corpus_dir": "/home/user/policy-analysis/Slavery_text",
        "dictionary_excel": "/home/user/policy-analysis/dutch_slavery_legacy_dictionary.xlsx",
        "workflow_base": "/home/user/policy-analysis/workflow_data",
        
        # For loading pretrained models
        "pretrained_model_path": None,  # Set to path of pretrained model if using one
    },
    
    # =====================
    # MODEL SETTINGS
    # =====================
    "model": {
        "base_model_name": "NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers",
        "use_pretrained": False,  # Set to True to load a previously fine-tuned model
    },
    
    # =====================
    # DICTIONARY SETTINGS
    # =====================
    "dictionary": {
        "use_excel": True,
        "topic_column": "topic",
        "keyword_column": "keyword",
        "sheet_name": 0,
        
        # Default seed topics (used if Excel not found)
        "default_topics": {
            "Historical slavery": ["slavernij", "tot-slaaf-gemaakte", "dwangarbeid", "zweep"],
            "Colonialism": ["kolonie", "koloniaal", "voc", "wic", "exploitatie"],
            "Modern racism& inequality": ["racisme", "discriminatie", "ongelijkheid"],
        },
    },
    
    # =====================
    # TEXT PROCESSING
    # =====================
    "chunking": {
        "sentences_per_chunk": 10,
        "min_sentences_to_keep": 3,
        "drop_likely_english": True,
        "remove_stopwords": True,
        "use_stemming": False,
    },
    
    "tokenize": {
        "lower": True,
        "keep_hyphen": True,
        "min_len": 2,
        "max_len": 30,
        "pattern": r"[0-9A-Za-zÀ-ÖØ-öø-ÿ\-]+",
    },
    
    # =====================
    # VOCABULARY SETTINGS
    # =====================
    "vocab": {
        "min_df": 5,          # Minimum document frequency
        "max_vocab": 50000,   # Maximum vocabulary size
    },
    
    # =====================
    # EXPANSION SETTINGS
    # =====================
    "expand": {
        "k_nearest": 50,
        "topN_per_topic": 300,
        "min_cosine": 0.55,
    },
    
    # =====================
    # SCORING SETTINGS
    # =====================
    "scoring": {
        "use_sif": True,
        "sif_a": 1e-3,
        
        # Confidence thresholds
        "high_confidence_score": 0.50,
        "high_confidence_margin": 0.05,
        "low_confidence_score": 0.40,
        "low_confidence_margin": 0.02,
    },
    
    # =====================
    # TRAINING SETTINGS
    # =====================
    "training": {
        "num_epochs": 3,
        "batch_size_train": 16,
        "batch_size_eval": 32,
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        
        # Dataset option: "option1", "option2", "option3", "option4"
        "dataset_option": "option4",  # Use comprehensive dataset by default
    },
}

print("✓ Configuration loaded")
print(f"\nWorkflow: {CONFIG['workflow']['model_type']}-{CONFIG['workflow']['topic']}")

In [ ]:
# ============================================================
# CELL 0.3: FILE SYSTEM UTILITIES
# ============================================================

class WorkflowFileSystem:
    """Manages the structured folder system for workflow data."""
    
    def __init__(self, config):
        self.config = config
        self.root = None
        self.folders = {}
    
    def create_workflow_folder(self):
        """Create the main workflow folder with structured subfolders."""
        # Generate folder name
        model_type = self.config["workflow"]["model_type"]
        topic = self.config["workflow"]["topic"]
        date = datetime.now().strftime("%m.%d.%y")
        
        # Handle version
        version = self.config["workflow"]["version"]
        if version is None:
            version = self._get_next_version(model_type, topic, date)
        
        folder_name = f"{model_type}-{topic}_{date}_{version}"
        
        # Create root folder
        base_dir = self.config["paths"]["workflow_base"]
        self.root = Path(base_dir) / folder_name
        self.root.mkdir(parents=True, exist_ok=True)
        
        # Create subfolders
        subfolder_names = [
            "config",
            "Dictionary",
            "Dictionary/Dictionary_suggestions",
            "Model_finetuning",
            "Cosine_labeling",
            "Bertje_labeling",
            "Visuals",
            "Other_data",
        ]
        
        for subfolder in subfolder_names:
            path = self.root / subfolder
            path.mkdir(parents=True, exist_ok=True)
            # Store without nested path for easy access
            key = subfolder.split("/")[-1]  # Get last part of path
            self.folders[key] = path
        
        # Also store Dictionary root for easy access
        self.folders["Dictionary"] = self.root / "Dictionary"
        
        print(f"\n{'='*60}")
        print("WORKFLOW FOLDER CREATED")
        print(f"{'='*60}")
        print(f"Location: {self.root}")
        print(f"\nSubfolders created:")
        for name in subfolder_names:
            print(f"  ✓ {name}/")
        
        return self.root
    
    def _get_next_version(self, model_type, topic, date):
        """Auto-increment version number."""
        base_dir = Path(self.config["paths"]["workflow_base"])
        if not base_dir.exists():
            return "v1"
        
        # Find existing folders with same prefix
        prefix = f"{model_type}-{topic}_{date}_v"
        existing = [d.name for d in base_dir.iterdir() if d.is_dir() and d.name.startswith(prefix)]
        
        if not existing:
            return "v1"
        
        # Extract version numbers
        versions = []
        for folder in existing:
            try:
                version_str = folder.split("_v")[-1]
                versions.append(int(version_str.replace("v", "")))
            except:
                continue
        
        if versions:
            next_num = max(versions) + 1
            return f"v{next_num}"
        return "v1"
    
    def load_existing_workflow(self, folder_path):
        """Load an existing workflow folder."""
        self.root = Path(folder_path)
        if not self.root.exists():
            raise ValueError(f"Workflow folder not found: {folder_path}")
        
        # Map subfolders
        subfolder_names = [
            "config",
            "Dictionary",
            "Dictionary_suggestions",
            "Model_finetuning",
            "Cosine_labeling",
            "Bertje_labeling",
            "Visuals",
            "Other_data",
        ]
        
        for name in subfolder_names:
            if name == "Dictionary_suggestions":
                path = self.root / "Dictionary" / name
            else:
                path = self.root / name
            
            if path.exists():
                self.folders[name] = path
        
        print(f"✓ Loaded existing workflow: {self.root.name}")
        return self.root
    
    def save_config(self, checkpoint_name=None):
        """Save current CONFIG to the config folder."""
        if self.root is None:
            raise ValueError("Workflow folder not initialized")
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        if checkpoint_name:
            filename = f"config_{checkpoint_name}_{timestamp}.json"
        else:
            filename = f"config_{timestamp}.json"
        
        config_path = self.folders["config"] / filename
        
        config_data = {
            "metadata": {
                "timestamp": timestamp,
                "checkpoint": checkpoint_name,
                "workflow_folder": str(self.root),
            },
            "config": self.config
        }
        
        with open(config_path, 'w', encoding='utf-8') as f:
            json.dump(config_data, f, indent=2, ensure_ascii=False)
        
        print(f"✓ Config saved: {config_path.name}")
        return config_path
    
    def save_data(self, data, filename, folder_key, file_format="csv"):
        """Save data to a specific folder.
        
        Args:
            data: DataFrame, dict, or numpy array
            filename: Name of the file (without extension)
            folder_key: Key from self.folders dict
            file_format: "csv", "json", "npy"
        """
        if self.root is None:
            raise ValueError("Workflow folder not initialized")
        
        folder = self.folders.get(folder_key)
        if folder is None:
            raise ValueError(f"Unknown folder key: {folder_key}")
        
        # Add extension
        full_filename = f"{filename}.{file_format}"
        filepath = folder / full_filename
        
        # Save based on format
        if file_format == "csv":
            if isinstance(data, pd.DataFrame):
                data.to_csv(filepath, index=False, encoding='utf-8')
            else:
                raise ValueError("CSV format requires DataFrame")
        
        elif file_format == "json":
            with open(filepath, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
        
        elif file_format == "npy":
            np.save(filepath, data, allow_pickle=True)
        
        else:
            raise ValueError(f"Unsupported format: {file_format}")
        
        print(f"✓ Saved: {folder_key}/{full_filename}")
        return filepath
    
    def copy_file_to_folder(self, source_path, folder_key, new_name=None):
        """Copy an external file to a workflow folder."""
        if self.root is None:
            raise ValueError("Workflow folder not initialized")
        
        folder = self.folders.get(folder_key)
        if folder is None:
            raise ValueError(f"Unknown folder key: {folder_key}")
        
        source = Path(source_path)
        if not source.exists():
            raise FileNotFoundError(f"Source file not found: {source_path}")
        
        dest_name = new_name if new_name else source.name
        dest_path = folder / dest_name
        
        shutil.copy2(source, dest_path)
        print(f"✓ Copied: {source.name} → {folder_key}/{dest_name}")
        return dest_path

print("✓ WorkflowFileSystem class defined")

In [ ]:
# ============================================================
# CELL 0.4: CREATE OR LOAD WORKFLOW
# ============================================================

# Choose one:
CREATE_NEW = True  # Set to False to load existing workflow
EXISTING_FOLDER = None  # Set to folder path if loading existing

# Initialize file system
fs = WorkflowFileSystem(CONFIG)

if CREATE_NEW:
    workflow_root = fs.create_workflow_folder()
    fs.save_config("initial_setup")
else:
    if EXISTING_FOLDER is None:
        raise ValueError("EXISTING_FOLDER must be set when CREATE_NEW=False")
    workflow_root = fs.load_existing_workflow(EXISTING_FOLDER)

print(f"\n✓ Workflow initialized: {workflow_root}")
print(f"\nAvailable folders:")
for key, path in fs.folders.items():
    print(f"  {key}: {path}")

In [ ]:
# ============================================================
# CELL 0.5: LOAD DICTIONARY
# ============================================================

def load_dictionary_from_excel(excel_path, config):
    """Load topics and keywords from Excel file."""
    if not Path(excel_path).exists():
        print(f"⚠ Excel file not found: {excel_path}")
        print("  Using default topics from config")
        return config["dictionary"]["default_topics"]
    
    try:
        df = pd.read_excel(
            excel_path, 
            sheet_name=config["dictionary"]["sheet_name"]
        )
        
        topic_col = config["dictionary"]["topic_column"]
        keyword_col = config["dictionary"]["keyword_column"]
        
        if topic_col not in df.columns or keyword_col not in df.columns:
            print(f"⚠ Required columns not found")
            return config["dictionary"]["default_topics"]
        
        topics_dict = {}
        for topic, group in df.groupby(topic_col):
            keywords = group[keyword_col].dropna().str.strip().tolist()
            if keywords:
                topics_dict[topic] = keywords
        
        print(f"✓ Loaded topics from Excel: {Path(excel_path).name}")
        print(f"  Topics: {len(topics_dict)}, Keywords: {len(df)}")
        
        return topics_dict
        
    except Exception as e:
        print(f"⚠ Error loading Excel: {e}")
        return config["dictionary"]["default_topics"]

# Load dictionary
if CONFIG["dictionary"]["use_excel"]:
    topics = load_dictionary_from_excel(
        CONFIG["paths"]["dictionary_excel"],
        CONFIG
    )
    CONFIG["topics"] = topics
    
    # Copy dictionary to workflow folder
    if Path(CONFIG["paths"]["dictionary_excel"]).exists():
        fs.copy_file_to_folder(
            CONFIG["paths"]["dictionary_excel"],
            "Dictionary",
            "input_dictionary.xlsx"
        )
else:
    CONFIG["topics"] = CONFIG["dictionary"]["default_topics"]
    print("✓ Using default topics from config")

print(f"\n{'='*60}")
print("TOPICS LOADED")
print(f"{'='*60}")
for topic, keywords in CONFIG["topics"].items():
    print(f"  {topic}: {len(keywords)} keywords")

# Save config with topics
fs.save_config("with_dictionary")

✅ **CHECKPOINT 0 COMPLETE**

You can now proceed to:
- **CHECKPOINT 1**: Text Processing
- Or skip to later checkpoints if you have existing data